# 02 – Data Collection with Web Scraping

**Goal:** scrape the Falcon 9 launch tables from Wikipedia and save them as `data/spacex_web_scraped.csv`.

We use the **fixed old revision** used by the IBM lab (`oldid=1027686922`, June 2021), so the result is repeatable.

**Steps:** request the HTML page → parse it with BeautifulSoup → find the launch tables → read each row → build a DataFrame → save.

In [1]:
import re, unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
HEADERS = {"User-Agent": "spacex-capstone-student/1.0"}

## 1. Request the page and create a BeautifulSoup object

In [2]:
response = requests.get(URL, headers=HEADERS, timeout=60)
print("status code:", response.status_code)
soup = BeautifulSoup(response.text, "html.parser")
print("page title:", soup.title.string)

status code: 200


page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


## 2. Find the launch tables and read the column names

In [3]:
html_tables = soup.find_all("table", "wikitable plainrowheaders collapsible")
print("launch tables found:", len(html_tables))

def clean(text):
    """Remove footnote marks like [12] and extra spaces."""
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"\[\s*[\w ]+\s*\]", "", text)
    return " ".join(text.split())

column_names = [clean(th.get_text(" ")) for th in html_tables[0].find_all("th")][:10]
column_names

launch tables found: 9


['Flight No.',
 'Date and time ( UTC )',
 'Version, Booster',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome',
 'Booster landing']

## 3. Helper functions to read each cell

In [4]:
def date_time(cell):
    """'4 June 2010, 18:45' -> ['4 June 2010', '18:45']"""
    parts = [s.strip() for s in cell.strings if s.strip()]
    return parts[0].strip(","), (parts[1] if len(parts) > 1 else None)

def get_mass(cell):
    """'4,700 kg' -> 4700.0 (None if no number)"""
    m = re.search(r"([\d,\.]+)\s*kg", clean(cell.get_text(" ")))
    return float(m.group(1).replace(",", "")) if m else None

def first_link_or_text(cell):
    return clean(cell.a.get_text()) if cell.a else clean(cell.get_text(" "))

## 4. Read every launch row

In [5]:
rows = []
for table in html_tables:
    for tr in table.find_all("tr"):
        th = tr.th
        # a launch row starts with the flight number in a <th> cell
        if not (th and th.string and th.string.strip().isdigit()):
            continue
        td = tr.find_all("td")
        if len(td) < 9:
            continue
        date, time = date_time(td[0])
        rows.append({
            "Flight No.": int(th.string.strip()),
            "Date": date,
            "Time": time,
            "Version Booster": clean(td[1].get_text(" ")),
            "Launch site": first_link_or_text(td[2]),
            "Payload": first_link_or_text(td[3]),
            "Payload mass (kg)": get_mass(td[4]),
            "Orbit": first_link_or_text(td[5]),
            "Customer": first_link_or_text(td[6]),
            "Launch outcome": clean(td[7].get_text(" ")),
            "Booster landing": clean(td[8].get_text(" ")),
        })

df = pd.DataFrame(rows)
df["Date"] = pd.to_datetime(df["Date"], format="%d %B %Y")
print("rows:", df.shape[0], "| columns:", df.shape[1])
df.head()

rows: 121 | columns: 11


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass (kg),Orbit,Customer,Launch outcome,Booster landing
0,1,2010-06-04,18:45,F9 v1.0 B0003.1,CCAFS,Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute)
1,2,2010-12-08,15:43,F9 v1.0 B0004.1,CCAFS,Dragon,NaN,LEO,NASA,Success,Failure (parachute)
2,3,2012-05-22,07:44,F9 v1.0 B0005.1,CCAFS,Dragon,525.0,LEO,NASA,Success,No attempt
3,4,2012-10-08,00:35,F9 v1.0 B0006.1,CCAFS,SpaceX CRS-1,4700.0,LEO,NASA,Success,No attempt
4,5,2013-03-01,15:10,F9 v1.0 B0007.1,CCAFS,SpaceX CRS-2,4877.0,LEO,NASA,Success,No attempt


## 5. Quick look at the scraped data

In [6]:
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print()
print(df["Launch site"].value_counts())
print()
print(df["Launch outcome"].value_counts())

Date range: 2010-06-04 to 2021-06-06

Launch site
CCAFS             40
KSC               33
Cape Canaveral    20
VAFB              16
CCSFS             12
Name: count, dtype: int64

Launch outcome
Success                120
Failure (in flight)      1
Name: count, dtype: int64


In [7]:
# first word of the landing cell = landing result
df["Booster landing"].str.split().str[0].value_counts()

Booster landing
Success         80
No              22
Failure         11
Controlled       5
Uncontrolled     2
Precluded        1
Name: count, dtype: int64

## 6. Save

In [8]:
df.to_csv("../data/spacex_web_scraped.csv", index=False)
print("saved ../data/spacex_web_scraped.csv", df.shape)

saved ../data/spacex_web_scraped.csv (121, 11)
